# SynBioCrow 2.4.13.15 — MIBK Isolated Subprocess Repair
Runs MIBK only. DORAnet and RetroBioCat2 are executed in short-lived child processes that persist candidate objects to disk and exit before the next backend starts. The parent process never imports the heavy backend stack. Targeted blockers are the cycle-confirmed development-only blockers from 2.4.13.13. Frozen sink panel and validation firewall remain unchanged.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, zipfile, gc

drive.mount("/content/drive", force_remount=False)
T0=time.time()
REPO=Path("/content/SynBioCrow_2_4_13_15")
WORK=Path("/content/synbiocrow_2_4_13_15")
OUT=Path("/content/drive/MyDrive/SynBioCrow/2.4/mibk_2_4_13_15")
for p in (WORK,OUT): p.mkdir(parents=True,exist_ok=True)

def status(p,m):
    print(f"[2.4.13.15] {p:5.1f}% | elapsed {(time.time()-T0)/60:5.1f} min | {m}",flush=True)

def run_stream(cmd,cwd=None,timeout=3600):
    print("$"," ".join(map(str,cmd)),flush=True)
    env=os.environ.copy()
    if cwd:
        env["PYTHONPATH"]=str(cwd)+os.pathsep+env.get("PYTHONPATH","")
    p=subprocess.Popen(list(map(str,cmd)),cwd=cwd,env=env,text=True,
                       stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                       bufsize=1)
    start=time.time()
    lines=[]
    for line in p.stdout:
        print(line,end="",flush=True); lines.append(line)
        if time.time()-start > timeout:
            p.kill(); raise TimeoutError(cmd)
    rc=p.wait()
    if rc:
        raise subprocess.CalledProcessError(rc,cmd,output="".join(lines))
    return "".join(lines)

status(5,"clone develop/2.4")
shutil.rmtree(REPO,ignore_errors=True)
run_stream(["git","clone","--depth","1","--branch","develop/2.4",
            "https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)

status(12,"install core + DORAnet")
run_stream([sys.executable,"-m","pip","install","-q","-e",str(REPO)+"[doranet]"],timeout=900)
status(18,"install pinned RetroBioCat2 without importing it in parent")
run_stream([sys.executable,"-m","pip","install","-q","PyYAML","sqlalchemy","tables","gdown","tqdm"],timeout=900)
RBC2_REV="c5f32561b1ed99d0701a6dbb472dc448d6c082e8"
run_stream([sys.executable,"-m","pip","install","-q","--no-deps","--no-build-isolation",
            f"git+https://github.com/willfinnigan/RetroBioCat-2.git@{RBC2_REV}"],timeout=1200)
run_stream([sys.executable,"-m","pip","install","-q","-e",str(REPO)],timeout=600)

TARGET="CC(C)CC(=O)C"
SINK_PANEL=REPO/"benchmarks/2.4/development_sink_panel_2_4_13_1.json"
WORKER=REPO/"scripts/v24_isolated_backend_generate_2_4_13_15.py"
EVAL=REPO/"scripts/v24_evaluate_mibk_isolated_2_4_13_15.py"

def generate(tag,backend,target,quick=False):
    pk=WORK/f"{tag}.pkl"
    js=WORK/f"{tag}.json"
    cmd=[sys.executable,str(WORKER),
         "--backend",backend,"--target",target,
         "--sink-panel",str(SINK_PANEL),
         "--pickle-out",str(pk),
         "--summary-out",str(js)]
    if quick: cmd.append("--quick")
    run_stream(cmd,cwd=REPO,timeout=3600)
    assert pk.is_file() and js.is_file()
    gc.collect()
    return pk,js

status(25,"isolated DORAnet MIBK generation")
dora_pk,dora_js=generate("initial_doranet","doranet",TARGET,False)

status(35,"isolated RetroBioCat2 MIBK generation")
rbc_pk,rbc_js=generate("initial_retrobiocat2","retrobiocat2",TARGET,False)

# Cycle-confirmed development blockers from 2.4.13.13.
BLOCKERS=[
    ("b1_amino","CC(C)CC(C)N"),
    ("b2_enone","CC(=O)C=C(C)C"),
    ("b3_alcohol","CC(C)CC(C)O"),
]
pickles=[dora_pk,rbc_pk]
summaries=[dora_js,rbc_js]
step=42
for label,smi in BLOCKERS:
    for backend in ("doranet","retrobiocat2"):
        status(step,f"isolated targeted expansion {backend} {smi}")
        pk,js=generate(f"{label}_{backend}",backend,smi,True)
        pickles.append(pk); summaries.append(js)
        step+=6

status(80,"lightweight merge + closure evaluation")
result=WORK/"mibk_isolated_evaluation.json"
cmd=[sys.executable,str(EVAL),
     "--target",TARGET,
     "--sink-panel",str(SINK_PANEL)]
for pk in pickles:
    cmd += ["--candidate-pickle",str(pk)]
cmd += ["--out",str(result)]
run_stream(cmd,cwd=REPO,timeout=1200)
evaluation=json.loads(result.read_text())
assert evaluation["frozen_sink_panel_unchanged"] is True
assert evaluation["validation_truth_accessed"] is False
assert evaluation["tuning_performed"] is False
print("MIBK strict routes:",evaluation["strict_route_count"],flush=True)
print("MIBK mixed admissible:",evaluation["mixed_boundary"]["admissible_route_count"],flush=True)
print("MIBK min unresolved:",evaluation["near_miss"]["minimum_unresolved_noncurrency_leaf_count"],flush=True)

status(90,"persist artifacts")
for p in summaries+[result]:
    shutil.copy2(p,OUT/p.name)
manifest={
    "version":"2.4.13.15",
    "target":"methyl isobutyl ketone",
    "target_smiles":TARGET,
    "blockers":[{"label":x,"smiles":y} for x,y in BLOCKERS],
    "isolated_backend_processes":True,
    "parent_backend_imports":False,
    "frozen_sink_panel_unchanged":True,
    "validation_truth_accessed":False,
    "tuning_performed":False,
}
(OUT/"manifest.json").write_text(json.dumps(manifest,indent=2,sort_keys=True)+"\n")
bundle=OUT/"SynBioCrow_2_4_13_15_MIBK_ISOLATED_SUBPROCESS_REPAIR_ALL_OUTPUTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file() and p!=bundle: z.write(p,p.name)
status(100,"PASS")
print("bundle:",bundle)
print("sha256:",hashlib.sha256(bundle.read_bytes()).hexdigest())
files.download(str(bundle))


## Rescue download


In [ ]:
from google.colab import files
from pathlib import Path
p=Path("/content/drive/MyDrive/SynBioCrow/2.4/mibk_2_4_13_15/SynBioCrow_2_4_13_15_MIBK_ISOLATED_SUBPROCESS_REPAIR_ALL_OUTPUTS.zip")
assert p.is_file(),p
files.download(str(p))
